# Notebook 3: Spark SQL and Temporary Views

## Goal

The goal of this notebook is to learn how Spark lets you mix Python DataFrame operations and SQL.

Spark SQL allows you to query structured data using either:

1. The PySpark DataFrame API
2. SQL queries through `spark.sql()`

Both approaches use the same underlying Spark execution engine.

That means this DataFrame code:

```python
transactions_df.groupBy("store_id").sum("amount")
```
and this SQL code:
```
SELECT store_id, SUM(amount)
FROM transactions
GROUP BY store_id
```
are two different ways of expressing similar logic.
This is useful because some users prefer Python, while others prefer SQL. In Databricks, you will often use both.


## What You Will Learn

| Topic | Practice |
|---|---|
| Create temp views | `createOrReplaceTempView()` |
| Query with SQL | `spark.sql()` |
| Compare SQL vs DataFrame syntax | Same task both ways |
| Use SQL aggregations | `GROUP BY`, `ORDER BY` |
| Use SQL filters | `WHERE`, `HAVING` |

## Key Lesson

Spark SQL and PySpark DataFrames are two interfaces to the same engine.

For Databricks, analytics, data engineering, and Solution Architect work, being fluent in both is valuable.


# Section 1: Setup

In [0]:
# Import common Spark SQL functions.
# We will use these for the DataFrame API examples.

from pyspark.sql import functions as F
from pyspark.sql.types import (
   StructType,
   StructField,
   IntegerType,
   StringType,
   DoubleType,
   TimestampType
)
from datetime import datetime

In [0]:
# Check that the SparkSession exists.
# In Databricks, spark is usually created automatically.

spark

In [0]:
# Print the Spark version.

print("Spark version:", spark.version)

Spark version: 4.1.0


# Section 2: Create a Retail Transaction DataFrame

## 1. Create a Retail Transaction DataFrame

We will create a small retail transaction dataset.

Each row represents one transaction.

The dataset includes:

| Column | Meaning |
|---|---|
| transaction_id | Unique transaction identifier |
| store_id | Store where the transaction happened |
| customer_id | Customer identifier |
| amount | Transaction amount |
| category | Product category |
| payment_method | Payment method used |
| timestamp | Time of transaction |

This dataset will let us practice both DataFrame operations and SQL queries.

In [0]:
# Create sample transaction data.
# Each tuple is one transaction.

transaction_data = [
   (1001, "Store_001", "Customer_001", 249.99, "Electronics", "Credit Card", datetime(2026, 5, 1, 9, 15, 0)),
   (1002, "Store_001", "Customer_002", 34.50, "Grocery", "Cash", datetime(2026, 5, 1, 10, 5, 0)),
   (1003, "Store_002", "Customer_003", 89.99, "Clothing", "Debit Card", datetime(2026, 5, 1, 11, 30, 0)),
   (1004, "Store_002", "Customer_001", 599.99, "Electronics", "Credit Card", datetime(2026, 5, 2, 14, 45, 0)),
   (1005, "Store_003", "Customer_004", 22.25, "Grocery", "Cash", datetime(2026, 5, 2, 16, 20, 0)),
   (1006, "Store_003", "Customer_005", 129.99, "Home Goods", "Credit Card", datetime(2026, 5, 3, 13, 10, 0)),
   (1007, "Store_001", "Customer_006", 14.99, "Grocery", "Debit Card", datetime(2026, 5, 3, 17, 55, 0)),
   (1008, "Store_002", "Customer_007", 799.99, "Electronics", "Credit Card", datetime(2026, 5, 4, 12, 0, 0)),
   (1009, "Store_003", "Customer_008", 45.00, "Clothing", "Cash", datetime(2026, 5, 4, 15, 35, 0)),
   (1010, "Store_001", "Customer_009", 199.99, "Home Goods", "Credit Card", datetime(2026, 5, 5, 18, 10, 0)),
   (1011, "Store_004", "Customer_010", 1099.99, "Electronics", "Credit Card", datetime(2026, 5, 5, 19, 25, 0)),
   (1012, "Store_004", "Customer_011", 64.75, "Grocery", "Debit Card", datetime(2026, 5, 6, 8, 45, 0))
]

In [0]:
# Define a manual schema.
# This gives us control over the column names and data types.

transaction_schema = StructType([
   StructField("transaction_id", IntegerType(), nullable=False),
   StructField("store_id", StringType(), nullable=False),
   StructField("customer_id", StringType(), nullable=False),
   StructField("amount", DoubleType(), nullable=False),
   StructField("category", StringType(), nullable=True),
   StructField("payment_method", StringType(), nullable=True),
   StructField("timestamp", TimestampType(), nullable=False)
])

In [0]:
# Create the Spark DataFrame.

transactions_df = spark.createDataFrame(
   transaction_data,
   schema=transaction_schema
)

display(transactions_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,Electronics,Credit Card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,Grocery,Cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,Clothing,Debit Card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,Electronics,Credit Card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,Grocery,Cash,2026-05-02T16:20:00.000Z
1006,Store_003,Customer_005,129.99,Home Goods,Credit Card,2026-05-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,Grocery,Debit Card,2026-05-03T17:55:00.000Z
1008,Store_002,Customer_007,799.99,Electronics,Credit Card,2026-05-04T12:00:00.000Z
1009,Store_003,Customer_008,45.0,Clothing,Cash,2026-05-04T15:35:00.000Z
1010,Store_001,Customer_009,199.99,Home Goods,Credit Card,2026-05-05T18:10:00.000Z


In [0]:
# Inspect the schema.

transactions_df.printSchema()

root
 |-- transaction_id: integer (nullable = false)
 |-- store_id: string (nullable = false)
 |-- customer_id: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- timestamp: timestamp (nullable = false)



# Section 3: Create a Temporary View

## 2. Create a Temporary View

To query a Spark DataFrame with SQL, we need to register it as a temporary view.

A temporary view is like a SQL table name that points to a DataFrame.

The key method is:

```python
createOrReplaceTempView()
```
For example:
```
transactions_df.createOrReplaceTempView("transactions")
```
After that, we can query the DataFrame using SQL:
```
spark.sql("SELECT * FROM transactions")
```
A temporary view only exists during the current Spark session. If the cluster or notebook session restarts, you usually need to recreate it.


In [0]:
# Register the DataFrame as a temporary SQL view.
# The view name will be "transactions".

transactions_df.createOrReplaceTempView("transactions")

In [0]:
# Query the temporary view using Spark SQL.
# spark.sql() returns a Spark DataFrame.

sql_df = spark.sql("""
   SELECT *
   FROM transactions
""")

display(sql_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,Electronics,Credit Card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,Grocery,Cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,Clothing,Debit Card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,Electronics,Credit Card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,Grocery,Cash,2026-05-02T16:20:00.000Z
1006,Store_003,Customer_005,129.99,Home Goods,Credit Card,2026-05-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,Grocery,Debit Card,2026-05-03T17:55:00.000Z
1008,Store_002,Customer_007,799.99,Electronics,Credit Card,2026-05-04T12:00:00.000Z
1009,Store_003,Customer_008,45.0,Clothing,Cash,2026-05-04T15:35:00.000Z
1010,Store_001,Customer_009,199.99,Home Goods,Credit Card,2026-05-05T18:10:00.000Z


In [0]:
# Confirm that the result of spark.sql() is a Spark DataFrame.

type(sql_df)

pyspark.sql.connect.dataframe.DataFrame

# Section 4: SQL SELECT vs DataFrame select()

## 3. SQL SELECT vs DataFrame select()

Now let's perform the same task two ways.

Task:

Select only these columns:

- transaction_id
- store_id
- amount
- category

First, we will use the DataFrame API.

Then, we will use Spark SQL.


In [0]:
# DataFrame API version.
# select() chooses specific columns.

selected_df_api = transactions_df.select(
   "transaction_id",
   "store_id",
   "amount",
   "category"
)

display(selected_df_api)

transaction_id,store_id,amount,category
1001,Store_001,249.99,Electronics
1002,Store_001,34.5,Grocery
1003,Store_002,89.99,Clothing
1004,Store_002,599.99,Electronics
1005,Store_003,22.25,Grocery
1006,Store_003,129.99,Home Goods
1007,Store_001,14.99,Grocery
1008,Store_002,799.99,Electronics
1009,Store_003,45.0,Clothing
1010,Store_001,199.99,Home Goods


In [0]:
# SQL version.
# This performs the same column selection using SQL syntax.

selected_df_sql = spark.sql("""
   SELECT
       transaction_id,
       store_id,
       amount,
       category
   FROM transactions
""")

display(selected_df_sql)

transaction_id,store_id,amount,category
1001,Store_001,249.99,Electronics
1002,Store_001,34.5,Grocery
1003,Store_002,89.99,Clothing
1004,Store_002,599.99,Electronics
1005,Store_003,22.25,Grocery
1006,Store_003,129.99,Home Goods
1007,Store_001,14.99,Grocery
1008,Store_002,799.99,Electronics
1009,Store_003,45.0,Clothing
1010,Store_001,199.99,Home Goods


## Observation

Both approaches return a Spark DataFrame.

The main difference is syntax.

The DataFrame API feels more like Python.

Spark SQL feels more like querying a database table.

Both are valid and both use Spark's execution engine.


# Section 5: SQL WHERE vs DataFrame filter()

## 4. SQL WHERE vs DataFrame filter()

Now let's filter rows.

Task:

Find transactions where `amount` is greater than 100.

We will do this using:

1. DataFrame API with `filter()`
2. SQL with `WHERE`

In [0]:
# DataFrame API version.
# filter() keeps rows that satisfy a condition.

high_value_df_api = transactions_df.filter(
   F.col("amount") > 100
)

display(high_value_df_api)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,Electronics,Credit Card,2026-05-01T09:15:00.000Z
1004,Store_002,Customer_001,599.99,Electronics,Credit Card,2026-05-02T14:45:00.000Z
1006,Store_003,Customer_005,129.99,Home Goods,Credit Card,2026-05-03T13:10:00.000Z
1008,Store_002,Customer_007,799.99,Electronics,Credit Card,2026-05-04T12:00:00.000Z
1010,Store_001,Customer_009,199.99,Home Goods,Credit Card,2026-05-05T18:10:00.000Z
1011,Store_004,Customer_010,1099.99,Electronics,Credit Card,2026-05-05T19:25:00.000Z


In [0]:
# SQL version.
# WHERE filters rows in SQL.

high_value_df_sql = spark.sql("""
   SELECT *
   FROM transactions
   WHERE amount > 100
""")

display(high_value_df_sql)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,Electronics,Credit Card,2026-05-01T09:15:00.000Z
1004,Store_002,Customer_001,599.99,Electronics,Credit Card,2026-05-02T14:45:00.000Z
1006,Store_003,Customer_005,129.99,Home Goods,Credit Card,2026-05-03T13:10:00.000Z
1008,Store_002,Customer_007,799.99,Electronics,Credit Card,2026-05-04T12:00:00.000Z
1010,Store_001,Customer_009,199.99,Home Goods,Credit Card,2026-05-05T18:10:00.000Z
1011,Store_004,Customer_010,1099.99,Electronics,Credit Card,2026-05-05T19:25:00.000Z


## Observation

The DataFrame API and SQL query are doing the same logical operation.

DataFrame API:

```python
transactions_df.filter(F.col("amount") > 100)
```
SQL:
```
SELECT *
FROM transactions
WHERE amount > 100
```
For simple filtering, either style is fine.


# Section 6: SQL Aggregations vs DataFrame groupBy()

## 5. Aggregations with GROUP BY

Aggregations summarize multiple rows into fewer rows.

Common aggregation questions include:

- What is total sales by store?
- What is average transaction amount by category?
- How many transactions used each payment method?

In the DataFrame API, we use:

```python
groupBy().agg()
```
In SQL, we use:
```
GROUP BY
```

In [0]:
# DataFrame API version.
# Calculate total sales and number of transactions by store.

sales_by_store_api = (
   transactions_df
   .groupBy("store_id")
   .agg(
       F.sum("amount").alias("total_sales"),
       F.count("*").alias("num_transactions")
   )
   .orderBy(F.col("total_sales").desc())
)

display(sales_by_store_api)

store_id,total_sales,num_transactions
Store_002,1489.97,3
Store_004,1164.74,2
Store_001,499.47,4
Store_003,197.24,3


In [0]:
# SQL version.
# Calculate total sales and number of transactions by store.

sales_by_store_sql = spark.sql("""
   SELECT
       store_id,
       SUM(amount) AS total_sales,
       COUNT(*) AS num_transactions
   FROM transactions
   GROUP BY store_id
   ORDER BY total_sales DESC
""")

display(sales_by_store_sql)

store_id,total_sales,num_transactions
Store_002,1489.97,3
Store_004,1164.74,2
Store_001,499.47,4
Store_003,197.24,3


## Key Pattern

This DataFrame API code:

```python
transactions_df.groupBy("store_id").agg(F.sum("amount"))
```
is conceptually similar to this SQL code:
```
SELECT store_id, SUM(amount)
FROM transactions
GROUP BY store_id
```
The syntax is different, but the logic is the same.


# Section 7: Aggregating by Category

## 6. Aggregating by Product Category

Now let's calculate sales by product category.

Task:

For each category, calculate:

- total sales
- average transaction amount
- number of transactions

We will do this both ways again.

In [0]:
# DataFrame API version.
# Calculate summary statistics by category.

sales_by_category_api = (
   transactions_df
   .groupBy("category")
   .agg(
       F.sum("amount").alias("total_sales"),
       F.avg("amount").alias("avg_transaction_amount"),
       F.count("*").alias("num_transactions")
   )
   .orderBy(F.col("total_sales").desc())
)

display(sales_by_category_api)

category,total_sales,avg_transaction_amount,num_transactions
Electronics,2749.96,687.49,4
Home Goods,329.98,164.99,2
Grocery,136.49,34.1225,4
Clothing,134.99,67.495,2


In [0]:
# SQL version.
# Calculate the same summary statistics by category.

sales_by_category_sql = spark.sql("""
   SELECT
       category,
       SUM(amount) AS total_sales,
       AVG(amount) AS avg_transaction_amount,
       COUNT(*) AS num_transactions
   FROM transactions
   GROUP BY category
   ORDER BY total_sales DESC
""")

display(sales_by_category_sql)

category,total_sales,avg_transaction_amount,num_transactions
Electronics,2749.96,687.49,4
Home Goods,329.98,164.99,2
Grocery,136.49,34.1225,4
Clothing,134.99,67.495,2


# Section 8: HAVING vs Filtering After Aggregation

## 7. SQL HAVING

`WHERE` filters rows before aggregation.

`HAVING` filters groups after aggregation.

For example:

```sql
WHERE amount > 100
```
filters individual transactions before grouping.
But:
```
HAVING SUM(amount) > 500
```
filters grouped results after total sales are calculated.
Task:
Find stores where total sales are greater than 500.


In [0]:
# DataFrame API version.
# First group by store, then filter the aggregated result.

stores_over_500_api = (
   transactions_df
   .groupBy("store_id")
   .agg(
       F.sum("amount").alias("total_sales"),
       F.count("*").alias("num_transactions")
   )
   .filter(F.col("total_sales") > 500)
   .orderBy(F.col("total_sales").desc())
)

display(stores_over_500_api)

store_id,total_sales,num_transactions
Store_002,1489.97,3
Store_004,1164.74,2


In [0]:
# SQL version.
# HAVING filters after GROUP BY.

stores_over_500_sql = spark.sql("""
   SELECT
       store_id,
       SUM(amount) AS total_sales,
       COUNT(*) AS num_transactions
   FROM transactions
   GROUP BY store_id
   HAVING SUM(amount) > 500
   ORDER BY total_sales DESC
""")

display(stores_over_500_sql)

store_id,total_sales,num_transactions
Store_002,1489.97,3
Store_004,1164.74,2


## WHERE vs HAVING

| Clause | Filters | Example |
|---|---|---|
| `WHERE` | Individual rows before aggregation | `WHERE amount > 100` |
| `HAVING` | Groups after aggregation | `HAVING SUM(amount) > 500` |

This distinction matters a lot in analytics work.

# Section 9: Using SQL Functions

## 8. Using SQL Functions

Spark SQL includes many familiar SQL functions.

For example, we can use:

- `ROUND()`
- `YEAR()`
- `MONTH()`
- `DAY()`
- `COUNT()`
- `SUM()`
- `AVG()`

Let's extract the year, month, and day from the transaction timestamp.

In [0]:
# DataFrame API version.
# Use Spark functions to extract date parts.

transactions_with_date_parts_api = (
   transactions_df
   .select(
       "transaction_id",
       "store_id",
       "amount",
       "timestamp",
       F.year("timestamp").alias("year"),
       F.month("timestamp").alias("month"),
       F.dayofmonth("timestamp").alias("day")
   )
)

display(transactions_with_date_parts_api)

transaction_id,store_id,amount,timestamp,year,month,day
1001,Store_001,249.99,2026-05-01T09:15:00.000Z,2026,5,1
1002,Store_001,34.5,2026-05-01T10:05:00.000Z,2026,5,1
1003,Store_002,89.99,2026-05-01T11:30:00.000Z,2026,5,1
1004,Store_002,599.99,2026-05-02T14:45:00.000Z,2026,5,2
1005,Store_003,22.25,2026-05-02T16:20:00.000Z,2026,5,2
1006,Store_003,129.99,2026-05-03T13:10:00.000Z,2026,5,3
1007,Store_001,14.99,2026-05-03T17:55:00.000Z,2026,5,3
1008,Store_002,799.99,2026-05-04T12:00:00.000Z,2026,5,4
1009,Store_003,45.0,2026-05-04T15:35:00.000Z,2026,5,4
1010,Store_001,199.99,2026-05-05T18:10:00.000Z,2026,5,5


In [0]:
# SQL version.
# Use SQL functions to extract date parts.

transactions_with_date_parts_sql = spark.sql("""
   SELECT
       transaction_id,
       store_id,
       amount,
       timestamp,
       YEAR(timestamp) AS year,
       MONTH(timestamp) AS month,
       DAY(timestamp) AS day
   FROM transactions
""")

display(transactions_with_date_parts_sql)

transaction_id,store_id,amount,timestamp,year,month,day
1001,Store_001,249.99,2026-05-01T09:15:00.000Z,2026,5,1
1002,Store_001,34.5,2026-05-01T10:05:00.000Z,2026,5,1
1003,Store_002,89.99,2026-05-01T11:30:00.000Z,2026,5,1
1004,Store_002,599.99,2026-05-02T14:45:00.000Z,2026,5,2
1005,Store_003,22.25,2026-05-02T16:20:00.000Z,2026,5,2
1006,Store_003,129.99,2026-05-03T13:10:00.000Z,2026,5,3
1007,Store_001,14.99,2026-05-03T17:55:00.000Z,2026,5,3
1008,Store_002,799.99,2026-05-04T12:00:00.000Z,2026,5,4
1009,Store_003,45.0,2026-05-04T15:35:00.000Z,2026,5,4
1010,Store_001,199.99,2026-05-05T18:10:00.000Z,2026,5,5


# Section 10: Creating Another Temporary View from a Result

## 9. Creating a Temporary View from a Query Result

The result of a SQL query is also a Spark DataFrame.

That means we can:

1. Run a SQL query
2. Store the result in a DataFrame
3. Register that result as another temporary view
4. Query the new view

This is useful for breaking complex logic into stages.

In [0]:
# Create a summarized DataFrame using SQL.

store_summary_df = spark.sql("""
   SELECT
       store_id,
       SUM(amount) AS total_sales,
       AVG(amount) AS avg_transaction_amount,
       COUNT(*) AS num_transactions
   FROM transactions
   GROUP BY store_id
""")

display(store_summary_df)

store_id,total_sales,avg_transaction_amount,num_transactions
Store_001,499.47,124.8675,4
Store_002,1489.97,496.6566666666667,3
Store_003,197.24,65.74666666666667,3
Store_004,1164.74,582.37,2


In [0]:
# Register the summary DataFrame as a new temporary view.

store_summary_df.createOrReplaceTempView("store_summary")

In [0]:
# Query the new temporary view.

top_store_summary_df = spark.sql("""
   SELECT *
   FROM store_summary
   WHERE total_sales > 500
   ORDER BY total_sales DESC
""")

display(top_store_summary_df)

store_id,total_sales,avg_transaction_amount,num_transactions
Store_002,1489.97,496.6566666666667,3
Store_004,1164.74,582.37,2


# Section 11: Inspect Query Plans

## 10. Inspecting Query Plans

Both SQL queries and DataFrame API operations are converted into Spark execution plans.

The `explain()` method lets us inspect those plans.

You do not need to understand every detail yet.

For now, remember this:

Spark SQL and DataFrame operations are both translated into plans that Spark can optimize and execute.

In [0]:
# Explain the DataFrame API plan.

sales_by_store_api.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [total_sales#11033 DESC NULLS LAST]
         +- PhotonGroupingAgg(keys=[store_id#10977], functions=[sum(amount#10979), count(1)])
            +- PhotonRowToColumnar
               +- LocalTableScan [store_id#10977, amount#10979]


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
# Explain the Spark SQL plan.

sales_by_store_sql.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [total_sales#11056 DESC NULLS LAST]
         +- PhotonGroupingAgg(keys=[store_id#10977], functions=[sum(amount#10979), count(1)])
            +- PhotonRowToColumnar
               +- LocalTableScan [store_id#10977, amount#10979]


== Photon Explanation ==
The query is fully supported by Photon.


Even though the code looked different, Spark turns both approaches into execution plans.

Later, when you study performance, you will learn how to read these plans more carefully.

For now, the main idea is that Spark is not simply executing Python or SQL line by line.

It is building and optimizing a distributed computation plan.

# Section 12: Practice Exercises

# Practice Exercises

Try these on your own.

## Exercise 1

Write a SQL query that returns all transactions from `Store_001`.

Then write the same logic using the DataFrame API.

## Exercise 2

Write a SQL query that returns all `Electronics` transactions with amount greater than 300.

Then write the same logic using the DataFrame API.

## Exercise 3

Calculate total sales by `payment_method`.

Do this with both SQL and the DataFrame API.

## Exercise 4

Calculate average transaction amount by `store_id` and `category`.

Do this with both SQL and the DataFrame API.

## Exercise 5

Find categories where total sales are greater than 200.

Use SQL with `HAVING`.

Then write the same logic using the DataFrame API.

# Section 13: Common Mistakes

# Common Mistakes

## Mistake 1: Forgetting to Create the Temporary View

Before running SQL against a DataFrame, you need something like:

```python
transactions_df.createOrReplaceTempView("transactions")
```
If you forget this, Spark SQL will not know what transactions refers to.

## Mistake 2: Using WHERE Instead of HAVING
Use WHERE before aggregation.
Use HAVING after aggregation.

Incorrect:
```
SELECT store_id, SUM(amount) AS total_sales
FROM transactions
GROUP BY store_id
WHERE SUM(amount) > 500
```

Correct:
```
SELECT store_id, SUM(amount) AS total_sales
FROM transactions
GROUP BY store_id
HAVING SUM(amount) > 500
```

## Mistake 3: Confusing SQL Strings with Python Variables
Inside a SQL string, you refer to table and column names.
Outside the SQL string, you use Python variables and DataFrame objects.

## Mistake 4: Thinking SQL and DataFrames Use Different Engines
In Spark, SQL and DataFrame operations are both translated into Spark execution plans.
The syntax is different, but the engine is the same.


# Section 14: Key Takeaways

# Key Takeaways

In this notebook, you learned how to use Spark SQL and temporary views.

The most important ideas are:

1. A Spark DataFrame can be registered as a temporary view.
2. `createOrReplaceTempView()` gives a DataFrame a SQL-queryable name.
3. `spark.sql()` lets you write SQL queries against Spark views and tables.
4. The result of `spark.sql()` is also a Spark DataFrame.
5. DataFrame API and Spark SQL are two interfaces to the same Spark engine.
6. `WHERE` filters rows before aggregation.
7. `HAVING` filters groups after aggregation.
8. Being fluent in both Python and SQL is valuable in Databricks workflows.

The core pattern is:

```text
Create Spark DataFrame
   ↓
Register as temporary view
   ↓
Query with Spark SQL
   ↓
Receive result as Spark DataFrame
   ↓
Continue with SQL or DataFrame API
```
